In [26]:
import numpy as np
from IPython.display import display

In [27]:
from pivotal_bwt import parse_and_combine
file_path_criteria = "criteria.csv"
file_path_elicitation = "wbt_results.csv"
dict_data = parse_and_combine(file_path_criteria, file_path_elicitation)
display(dict_data)

{'Monetary': {'criteria': {'Salary': {'min_value': 1200.0,
    'max_value': 2000.0,
    'value_function': <function pivotal_bwt.<lambda>(x)>,
    'best_comparisons': {'Training Budget': 1436.0},
    'worst_comparisons': {}},
   'Bonus': {'min_value': 0.0,
    'max_value': 5000.0,
    'value_function': <function pivotal_bwt.<lambda>(x)>,
    'best_comparisons': {},
    'worst_comparisons': {'Salary': 1568.0, 'Training Budget': 230.0}},
   'Training Budget': {'min_value': 0.0,
    'max_value': 2000.0,
    'value_function': <function pivotal_bwt.<lambda>(x)>,
    'best_comparisons': {},
    'worst_comparisons': {}}},
  'intraB': {'best_Salary_Paid Leave': {'type': 'best',
    'reference': 'Salary',
    'other': 'Paid Leave',
    'value': 1748.0},
   'worst_Paid Leave_Salary': {'type': 'worst',
    'reference': 'Paid Leave',
    'other': 'Salary',
    'value': 1624.0}},
  'intraW': {'best_Commute Time_Bonus': {'type': 'best',
    'reference': 'Commute Time',
    'other': 'Bonus',
    'valu

In [28]:
from pivotal_bwt import constraints_func
num_groups = len(dict_data)
num_criteria = sum(len(group_data['criteria']) for group_data in dict_data.values())

# Total number of weights (criteria weights + group weights + auxiliary variable z)
tot_number = num_criteria + num_groups + 1
# Initial guess: for all w_i + z (auxiliary variable)
x0 = np.ones(tot_number) / tot_number
display(constraints_func(x0, dict_data))

[np.float64(-2.2787193973634654),
 np.float64(-0.42888888888888893),
 np.float64(-0.7738888888888888),
 np.float64(-0.24943310657596368),
 np.float64(-1.1616161616161618),
 np.float64(-0.3988888888888889),
 np.float64(-0.5928888888888888),
 np.float64(-0.34874290348742903),
 np.float64(-0.8839567063981368),
 np.float64(-0.879048888888889),
 np.float64(-0.5688888888888888),
 np.float64(0.1111111111111111),
 np.float64(0.1111111111111111),
 np.float64(0.1111111111111111),
 np.float64(0.1111111111111111),
 np.float64(0.1111111111111111),
 np.float64(0.1111111111111111),
 np.float64(0.1111111111111111),
 np.float64(0.1111111111111111),
 np.float64(-0.33333333333333326)]

In [29]:
from pivotal_bwt import bwt
results = bwt(dict_data)
display(results["solver_result"])

Starting optimization...


     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 4.715549534481889e-16
           x: [ 2.675e-01  1.227e-01  1.107e-01  1.869e-01  1.670e-01
                1.463e-01  9.978e-01  9.956e-01  4.716e-16]
         nit: 2
         jac: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
                0.000e+00  0.000e+00  0.000e+00  1.000e+00]
        nfev: 21
        njev: 2
 multipliers: [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]

In [30]:
# print the weights
weights = results["criteria_weights"]
for group_name, group_data in dict_data.items():
    print(f"Group: {group_name}")
    for crit_name, crit_data in group_data['criteria'].items():
        # weights is structured as weights[group_name][crit_name], so use names instead of an 'index' key
        weight = weights.get(group_name, {}).get(crit_name)
        print(f"  Criterion: {crit_name}, Weight: {weight}")

Group: Monetary
  Criterion: Salary, Weight: 0.2675058299872229
  Criterion: Bonus, Weight: 0.12269465149171097
  Criterion: Training Budget, Weight: 0.11074462693442871
Group: Non-monetary
  Criterion: Paid Leave, Weight: 0.186892507799386
  Criterion: Remote Work Days, Weight: 0.16704378002614226
  Criterion: Commute Time, Weight: 0.1463495535560529


In [31]:
first_run_error = results["solver_result"]["x"][-1]
display(f"Error (z): {first_run_error}")

'Error (z): 4.715549534481889e-16'

In [32]:
def flatten_weights(weights, dict_data):
    x = []
    # Flatten criteria weights
    for group_name, group_data in dict_data.items():
        for crit_name in group_data['criteria'].keys():
            x.append(weights[group_name][crit_name])
    # Flatten group weights (if present)
    # (Assuming group weights are not in weights, but if they are, append them here)
    return np.array(x)

x0 = flatten_weights(results["criteria_weights"], dict_data)
display(x0)

array([0.26750583, 0.12269465, 0.11074463, 0.18689251, 0.16704378,
       0.14634955])

In [33]:
import pytensor.tensor as pt
import pymc as pm

def pytensor_constraints_func(x, dict_data, Z_max):
    cons = []
    group_indices = {}
    current_index = 0
    num_criteria = sum(len(group_data['criteria']) for group_data in dict_data.values())

    # Map group and criterion names to their indices in x
    for group_name, group_data in dict_data.items():
        criteria_in_group = list(group_data['criteria'].keys())
        group_indices[group_name] = {
            'start': current_index,
            'end': current_index + len(criteria_in_group),
            'criteria': criteria_in_group
        }
        current_index += len(criteria_in_group)

    tot_number = current_index
    z_index = tot_number  # Z is the last element in x
    Z = x[z_index]  # Extract Z from x

    # INTRA-GROUP CONSTRAINTS
    for group_name, group_data in dict_data.items():
        criteria_in_group = group_indices[group_name]['criteria']
        w_start = group_indices[group_name]['start']
        w_end = group_indices[group_name]['end']
        w = x[w_start:w_end]  # Weights for this group

        for crit, comparisons in group_data['criteria'].items():
            v_f = comparisons['value_function']
            # BEST COMPARISONS
            for other_crit, value in comparisons['best_comparisons'].items():
                i = criteria_in_group.index(crit)
                j = criteria_in_group.index(other_crit)
                v_f_other = group_data['criteria'][crit]['value_function']
                cons.append(Z - pt.abs(w[i] / w[j] - 1.0 / v_f_other(value)))
            # WORST COMPARISONS
            for other_crit, value in comparisons['worst_comparisons'].items():
                i = criteria_in_group.index(crit)
                j = criteria_in_group.index(other_crit)
                v_f_other = group_data['criteria'][other_crit]['value_function']
                cons.append(Z - pt.abs(v_f_other(value) - w[i] / w[j]))

    # INTER-GROUP CONSTRAINTS
    intraB = {}
    intraW = {}
    for group_data in dict_data.values():
        intraB.update(group_data['intraB'])
        intraW.update(group_data['intraW'])

    w_G = x[tot_number:tot_number + len(dict_data)]  # Group weights

    # BEST-BEST and BEST-WORST COMPARISONS
    for crit_name, comparison in intraB.items():
        ref_crit = comparison['reference']
        other_crit = comparison['other']
        ref_group = next(g for g, gd in dict_data.items() if ref_crit in gd['criteria'])
        other_group = next(g for g, gd in dict_data.items() if other_crit in gd['criteria'])
        v_f_other = dict_data[ref_group]['criteria'][ref_crit]['value_function']
        cons.append(Z - pt.abs(w_G[0] / w_G[1] - 1.0 / v_f_other(comparison['value'])))

    # WORST-BEST and WORST-WORST COMPARISONS
    for crit_name, comparison in intraW.items():
        ref_crit = comparison['reference']
        other_crit = comparison['other']
        ref_group = next(g for g, gd in dict_data.items() if ref_crit in gd['criteria'])
        other_group = next(g for g, gd in dict_data.items() if other_crit in gd['criteria'])
        v_f_other = dict_data[other_group]['criteria'][other_crit]['value_function']
        cons.append(Z - pt.abs(v_f_other(comparison['value']) - w_G[1] / w_G[-1]))

    # NEW CONSTRAINT: Z must be smaller than Z_max
    cons.append(Z_max - Z)

    return pt.stack(cons)


In [37]:
import pymc as pm
import numpy as np

# Run MCMC to get posterior samples
with pm.Model() as model:
    # Define your model (weights, Z, constraints, etc.)
    w_crit = pm.Dirichlet("w_crit", a=np.ones(num_criteria))
    w_group = pm.Dirichlet("w_group", a=np.ones(num_groups))
    Z = pm.Uniform("Z", lower=0, upper=first_run_error)
    x = pm.math.concatenate([w_crit, w_group, Z[None]])
    cons = pytensor_constraints_func(x, dict_data, Z_max=first_run_error)
    pm.Potential("constraints", pt.switch(pt.any(pt.gt(cons, 1e-6)), -np.inf, 0.0))

    trace = pm.sample(2000, tune=1000, chains=4, target_accept=0.9, random_seed=42)

# Extract samples: shape = (n_samples, n_weights + 1)
posterior_samples = trace.posterior["x"].to_numpy()
np.save("posterior_samples.npy", posterior_samples)  # Save for later use


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [w_crit, w_group, Z]


/home/simo/GitHub/Master-Thesis/.venv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 16 seconds.
There were 7958 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


KeyError: "No variable named 'x'. Variables on the dataset include ['chain', 'draw', 'w_crit_dim_0', 'w_crit', 'w_group_dim_0', 'w_group', 'Z']"

In [ ]:
# Load posterior samples (if saved)
posterior_samples = np.load("posterior_samples.npy")

# Example Monte Carlo loop
n_mc_runs = 1000
results = []

for _ in range(n_mc_runs):
    # Randomly select one set of weights from the posterior
    weights = np.random.choice(posterior_samples.shape[0])
    w_crit, w_group, Z = (
        posterior_samples[weights, :num_criteria],
        posterior_samples[weights, num_criteria:-1],
        posterior_samples[weights, -1]
    )

    # Use these weights in your simulation
    result = run_your_simulation(w_crit, w_group, Z)
    results.append(result)


KeyError: 0